**Integrantes:** Nicolás Tovar, Emilio Puga



# 📔 **Resolución**
Una vez que la nota sea entregada, encontrarás la resolución en este link de notion:

https://www.notion.so/Soluciones-MMIA-3era-cohorte-1ea0573743578016a1aac56aaf239271?source=copy_link

Recuerda que existen varias formar de solucionar las actividades propuestas en los laboratorios. Toma la resolución entregada como referencia. En caso de que el link de resolución no esté público, solicita acceso en el mismo link de notion

# Optimización de hiperparámetros con PCA y sin PCA para la predicción de pérdida de clientes en Empresas de Servicios de Telecomunicaciones (Churn) con KNN


Reducir las salidas y [deserciones de clientes](https://www.bain.com/insights/breaking-the-back-of-customer-churn/) se ha convertido en una alta prioridad para la mayoría de los proveedores de servicios de comunicaciones a medida que los mercados maduran y la competencia se intensifica.




En este documento usaremos una base de datos de una empresa de telecomunicaciones anónima [disponibilizada por IBM](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv).

El principal objetivo es crear un model de aprendizaje automático basado en KNN (similar al propuesto [aquí con SVM](https://github.com/mmcuri/ds_handson/blob/master/Telecom_Churn_Prediction.ipynb)) para predecir la pérdida o salida de clientes en una empresa de telecomunicaciones.


## Librerías

Importamos varias bibliotecas y módulos útiles para el análisis de datos y el aprendizaje automático en Python. A continuación se describe cada uno de ellos:

- `pandas as pd`: Importa la biblioteca Pandas y la renombra como "pd". Pandas es una biblioteca popular utilizada para la manipulación y análisis de datos en Python.

- `numpy as np`: Importa la biblioteca NumPy y la renombra como "np". NumPy es una biblioteca de Python utilizada para realizar operaciones matemáticas en matrices y vectores.

- `matplotlib.pyplot as plt`: Importa la biblioteca Matplotlib y el módulo pyplot y lo renombra como "plt". Matplotlib es una biblioteca utilizada para la visualización de datos en Python.

- `from sklearn.metrics import classification_report`: Importa la función classification_report desde el módulo metrics de la biblioteca scikit-learn. classification_report es una función que calcula y muestra un informe de clasificación detallado para un modelo de clasificación.

- `from sklearn.metrics import confusion_matrix`: Importa la función confusion_matrix desde el módulo metrics de la biblioteca scikit-learn. confusion_matrix es una función que calcula y muestra la matriz de confusión para un modelo de clasificación.

- `from sklearn.metrics import accuracy_score`: Importa la función accuracy_score desde el módulo metrics de la biblioteca scikit-learn. accuracy_score es una función que calcula y muestra la precisión de un modelo de clasificación.

- `from sklearn.metrics import roc_auc_score`: Importa la función roc_auc_score desde el módulo metrics de la biblioteca scikit-learn. roc_auc_score es una función que calcula el área bajo la curva ROC (AUC) para un modelo de clasificación.

- `from sklearn.preprocessing import StandardScaler, LabelEncoder`: Importa las clases StandardScaler y LabelEncoder desde el módulo preprocessing de la biblioteca scikit-learn. StandardScaler es una clase utilizada para estandarizar los datos y LabelEncoder es una clase utilizada para codificar las etiquetas de clase en números enteros.

- `from sklearn.model_selection import train_test_split`: Importa la función train_test_split desde el módulo model_selection de la biblioteca scikit-learn. train_test_split es una función que divide los datos en conjuntos de entrenamiento y prueba para su uso en el modelado predictivo.

En resumen, este código importa varias bibliotecas y módulos que son útiles para el análisis de datos y el aprendizaje automático en Python.


In [2]:
# importing libraries
import pandas as pd #excel sofisticado
import numpy as np #matlab
import matplotlib.pyplot as plt #plots
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split


## Base de datos

Este conjunto de datos contiene un total de 7043 clientes y 21 características de los mismos. De las entradas, 5174 son clientes activos y 1869 son clientes que la empresa ha perdido. Observe que el conjunto de datos está desbalanceado pues por cada cliente perdido existe casi 3 clientes activos. La variable de salida para nuestro modelo de machine learning será `Churn`.

In [3]:
# importamos dataset
DATA_PATH = "https://raw.githubusercontent.com/mtgca/datasets_public/main/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)
#print(df)
# vemos las primeras 5 filas
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### *Detalles de la base de datos*

*	`customerID` - Custumer unique identifier
*	`gender` - Customer gender - ['Female' 'Male']
*	`SeniorCitizen` - Elderly or retired person, a senior citizen is someone who has at least attained the age of 60 of 65 years
*	`Partner` - - ['No' 'Yes']
*	`Dependents` - If customer has dependents - ['No' 'Yes']
*	`Tenure` - Customer lifespan (in months)
*	`PhoneService` - - ['No' 'Yes']
*	`MultipleLines` - - ['No' 'No phone service' 'Yes']
*	`InternetService` - - ['No' 'No internet service' 'Yes']
*	`OnlineSecurity` - - ['No' 'No internet service' 'Yes']
*	`OnlineBackup` - - ['No' 'No internet service' 'Yes']
*	`DeviceProtection` - - ['No' 'No internet service' 'Yes']
*	`TechSupport` - - ['No' 'No internet service' 'Yes']
*	`StreamingTV` - - ['No' 'No internet service' 'Yes']
*	`StreamingMovies` -  - ['No' 'No internet service' 'Yes']
*	`Contract` - Type of contract - ['Month-to-month' 'One year' 'Two year']
*	`PaperlessBilling` - - ['No' 'Yes']
*	`PaymentMethod` - payment method - ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
*	`MonthlyCharges` - Monthly Recurring Charges
*	`TotalCharges` - Life time value
*	`Churn` - Churn value, the targer vector - ['No' 'Yes']





### *Limpieza del Dataset*

El código presenta una serie de operaciones de preprocesamiento de datos en una base de datos que contiene información sobre clientes de una compañía, incluyendo su género, edad, servicios contratados, pagos realizados, entre otros.

En primer lugar, se define la función get_df_size para imprimir el número de atributos y entradas de la base de datos.

Luego, se reemplazan los valores en blanco por NaN para facilitar el manejo de valores faltantes en la base de datos.

A continuación, se reemplazan los valores faltantes en la columna 'TotalCharges' por la mediana de esa columna y se convierte esa columna al tipo numérico.

Después, se elimina la columna 'customerID' ya que no es una característica relevante.

A continuación, se codifican las características binarias en 0's y 1's usando LabelEncoder y se imprimen las categorías correspondientes a cada una de estas características.


In [4]:
def get_df_size(df, header='Dataset dimensions'):
  print(header,
        '\n# Attributes: ', df.shape[1],
        '\n# Entries: ', df.shape[0],'\n')

get_df_size(df)

#df.info()

# reemplaza valores en blanco por NaN
df_clean = df.replace(r'^\s*$', np.nan, regex=True)

# Convert 'TotalCharges' to numeric before calculating the median.
# errors='coerce' will handle any non-numeric values by setting them to NaN.
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# reemplaza valores faltantes en TotalCharges por la mediana de TotalCharges.
total_charges_median = df_clean.TotalCharges.median()
df_clean['TotalCharges'].fillna(total_charges_median, inplace=True)
df_clean['TotalCharges'] = df_clean['TotalCharges'].apply(pd.to_numeric)

#CustomerID lo retiramos porque no es una característica
df_clean = df_clean.drop('customerID', axis=1)
df_clean.describe()

print("Churn No Instances: ", df_clean[df_clean['Churn'] == 'No'].shape[0])
print("Churn Yes Instances: ", df_clean[df_clean['Churn'] == 'Yes'].shape[0])

binary_feat = df_clean.nunique()[df_clean.nunique() == 2].keys().tolist()
numeric_feat = [col for col in df_clean.select_dtypes(['float','int']).columns.tolist() if col not in binary_feat]
categorical_feat = [ col for col in df_clean.select_dtypes('object').columns.to_list() if col not in binary_feat + numeric_feat ]
df_proc = df_clean.copy()
#Etiquetas para características binarias
le = LabelEncoder()
for i in binary_feat:
  df_proc[i] = le.fit_transform(df_proc[i])
  print(i, '\n', np.unique(df_proc[i].values))
#Dummy variables
df_proc = pd.get_dummies(df_proc, columns=categorical_feat)
get_df_size(df, header='Original dataset:')
get_df_size(df_proc, header='Processed dataset:')
df_proc.head()


Dataset dimensions 
# Attributes:  21 
# Entries:  7043 

Churn No Instances:  5174
Churn Yes Instances:  1869
gender 
 [0 1]
SeniorCitizen 
 [0 1]
Partner 
 [0 1]
Dependents 
 [0 1]
PhoneService 
 [0 1]
PaperlessBilling 
 [0 1]
Churn 
 [0 1]
Original dataset: 
# Attributes:  21 
# Entries:  7043 

Processed dataset: 
# Attributes:  41 
# Entries:  7043 



/tmp/ipykernel_1765/2470922283.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['TotalCharges'].fillna(total_charges_median, inplace=True)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,True,False,False,True,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,True,False,False,False,True,False,False,False,False,True
2,1,0,0,0,2,1,1,53.85,108.15,1,...,True,False,False,True,False,False,False,False,False,True
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,True,False,False,False,True,False,True,False,False,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,True,False,False,True,False,False,False,False,True,False


### *División en conjunto de entrenamiento y test*

En el código, se divide el conjunto de datos df_proc en características y variable objetivo X e y, respectivamente. Luego, se realiza una división de los datos en conjuntos de entrenamiento y prueba utilizando la función train_test_split de scikit-learn. Los conjuntos de entrenamiento y prueba se almacenan en X_train, X_test, y_train y y_test. La división se realiza de manera estratificada para asegurar que la proporción de instancias positivas y negativas en la variable objetivo se mantenga en ambos conjuntos. Se utiliza una proporción de 80/20 para los conjuntos de entrenamiento y prueba, respectivamente, y se fija una semilla aleatoria (random_state) en 42 para asegurar que los resultados sean reproducibles.

In [5]:
# dividimos df_proc en características y salida
X=df_proc.drop('Churn', axis=1) #features
y=df_proc['Churn'] #output, target

# Dividimos el conjunto de entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

### *Balanceo de datos*

Este código realiza el submuestreo (undersampling) de un conjunto de datos desbalanceado en términos de la variable objetivo. El submuestreo se utiliza para abordar el problema del desequilibrio de clases en el conjunto de datos, donde una o varias clases tienen muchas menos observaciones que las otras clases. En este caso, el modelo de aprendizaje automático puede estar sesgado hacia la clase mayoritaria, lo que puede resultar en un rendimiento deficiente en la clasificación de la clase minoritaria.

El submuestreo implica reducir el número de observaciones en la clase mayoritaria para equilibrar la distribución de las clases en el conjunto de datos. Esto puede mejorar el rendimiento del modelo en la clasificación de la clase minoritaria y reducir el sesgo hacia la clase mayoritaria.

La biblioteca imblearn se utiliza para realizar el submuestreo, y en este caso se importa la clase RandomUnderSampler. El objeto RandomUnderSampler se inicializa con random_state=1, lo que garantiza que los resultados serán reproducibles.

Luego, el submuestreo se realiza en el conjunto de entrenamiento (X_train, y_train) mediante el método fit_resample del objeto rus. Los datos submuestreados se almacenan en X_train_rus y y_train_rus.

Para evaluar el efecto del submuestreo, se utiliza la función get_df_size, que devuelve el tamaño del conjunto de datos. Primero se imprime el tamaño del conjunto de entrenamiento original (X_train, y_train) antes del submuestreo, y luego se imprime el tamaño del conjunto submuestreado (X_train_rus, y_train_rus) después de aplicar el submuestreo.

Finalmente, se utiliza la función np.unique para verificar que las categorías de la variable objetivo (y_train_rus) estén balanceadas. La función devuelve una tupla que contiene las categorías únicas y el número de instancias en cada categoría. Al utilizar return_counts=True, se garantiza que se devuelvan los recuentos de las categorías.

In [6]:
# submuestreo -> under sampling
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=1)
X_train_undersampled, y_train_undersampled = rus.fit_resample(X_train, y_train)
get_df_size(X_train, header='Before balancing:')
get_df_size(X_train_undersampled, header='After balancing:')

# verificamos que las categorias estén balanceadas
np.unique(y_train_undersampled, return_counts=True)


Before balancing: 
# Attributes:  40 
# Entries:  5634 

After balancing: 
# Attributes:  40 
# Entries:  2990 



(array([0, 1]), array([1495, 1495]))

## Modelo de Machine Learning

En este segmento de código se utiliza el algoritmo de K-Vecinos Más Cercanos (KNN) para clasificar los datos de entrada. Primero, se crea un objeto de Pipeline que combina varios pasos de preprocesamiento y entrenamiento del modelo. Los pasos incluyen:

`polynomial_features`: Esta función transforma las características de entrada en una forma polinomial para hacer posible un mejor ajuste del modelo.

`scaler`: Normaliza las características de entrada para que todas tengan una media cero y una varianza unitaria.

`knn`: Un clasificador KNN que utiliza un número especificado de vecinos para determinar la clase de un punto de datos.

Una vez que se define el objeto de Pipeline, se ajusta al conjunto de entrenamiento utilizando el método fit(). Luego se utiliza el modelo ajustado para predecir las clases de los puntos de datos en el conjunto de prueba y en el conjunto de entrenamiento. Esto se hace mediante el método predict(). Finalmente, se utiliza la métrica de precisión (accuracy_score) de la biblioteca sklearn.metrics para evaluar la precisión del modelo en el conjunto de prueba y en el conjunto de entrenamiento.

### K Nearest Neighbors sin optimización de hiperparámetros



In [7]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import sklearn.metrics as metrics
from sklearn.preprocessing import PolynomialFeatures

# pipeline
pipeline_knn = Pipeline([
    # ('polynomial_features', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()) #parámetros por default
])

# training
pipeline_knn.fit(X_train_undersampled, y_train_undersampled)

# evaluation on test
pred_test = pipeline_knn.predict(X_test)
f1score = metrics.f1_score(y_test, pred_test)
acc = metrics.accuracy_score(y_test, pred_test)
preci = metrics.precision_score(y_test, pred_test)
recall = metrics.recall_score(y_test, pred_test)

print("Para el modelo NO optimizado, métricas en el conjunto de test: recall = ", recall, ", F1 = ", f1score, ", acc = ", acc, ", precision = ", preci, )


Para el modelo NO optimizado, métricas en el conjunto de test: recall =  0.7727272727272727 , F1 =  0.5639024390243902 , acc =  0.6827537260468417 , precision =  0.44393241167434716


### Optimización de Hiperparámetros KNN con kfold CV sin PCA

Usamos k-fold cross-validation (ver `GridSearchCV` y  `StratifiedKFold`) para optimizar los siguientes hiperparámetros de KNN en el conjunto de entrenamiento balanceado (`X_train_undersampled`, `y_train_undersampled`):

1. Ponderación de vecinos: sin ponderación, ponderación del inverso de la distancia
2. Número de vecinos: rango 1 a 100.
3. Métricas de distancia: euclideana, manhattan, coseno

Para la optimización de hiperparámetros utilizamos como métrica al recall.

Una vez optimizados los hiperparámetros, reentrenamos el clasificador de kNN con los mejores hiperparámetros encontrados. Finalmente, evaluamos este último clasificador en el conjunto de test `X_test`, `y_test`.)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

#optimización
kfold = StratifiedKFold(n_splits=2, shuffle=True, random_state=45) #para nsplits=5 demora 35 min approx en google colab.

# pipeline
pipeline_knn = Pipeline([
     #('polynomial_features', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
param_grid = {'knn__weights': ['uniform', 'distance'], #uniform es sin ponderación, weighted knn
              'knn__n_neighbors': np.arange(1,101,1), #1 a 100.
              'knn__metric': ['euclidean', 'manhattan', 'cosine']}


search = GridSearchCV(pipeline_knn, param_grid, scoring='recall', cv=kfold, verbose=4)
result = search.fit(X_train_undersampled, y_train_undersampled)

print(f'Mejor recall en el conj. de validación: {result.best_score_} para {result.best_params_}')

#reentreno con mejores
pipeline_knn.set_params(knn__n_neighbors=result.best_params_['knn__n_neighbors'],
                       knn__weights=result.best_params_['knn__weights'],
                       knn__metric=result.best_params_['knn__metric'])
pipeline_knn.fit(X_train_undersampled, y_train_undersampled)

#evaluamos en el conjunto de test
pred_test = pipeline_knn.predict(X_test)
f1score = metrics.f1_score(y_test, pred_test)
acc = metrics.accuracy_score(y_test, pred_test)
preci = metrics.precision_score(y_test, pred_test)
recall = metrics.recall_score(y_test, pred_test)

print("Para el modelo optimizado, métricas en el conjunto de test: recall = ", recall, ", F1 = ", f1score, ", acc = ", acc, ", precision = ", preci, )


Fitting 2 folds for each of 600 candidates, totalling 1200 fits
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=uniform;, score=0.704 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=uniform;, score=0.654 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=distance;, score=0.704 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=distance;, score=0.654 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=uniform;, score=0.538 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=uniform;, score=0.503 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=distance;, score=0.704 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=distance;, score=0.654 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors

### Optimización de hiperparámetros kNN con PCA

Repetiremos el proceso anterior pero esta vez con PCA

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

#optimización
kfold = StratifiedKFold(n_splits=2, shuffle=True, random_state=45) #para nsplits=5 demora 35 min approx en google colab.

# pipeline
pipeline_knn_pca = Pipeline([
     #('polynomial_features', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('knn', KNeighborsClassifier())
])
param_grid = {'knn__weights': ['uniform', 'distance'],
              'knn__n_neighbors': np.arange(1,101,1),
              'knn__metric': ['euclidean', 'manhattan', 'cosine']}


search = GridSearchCV(pipeline_knn_pca, param_grid, scoring='recall', cv=kfold, verbose=4)
result = search.fit(X_train_undersampled, y_train_undersampled)

print(f'Mejor recall en el conj. de validación: {result.best_score_} para {result.best_params_}')

#reentreno con mejores
pipeline_knn_pca.set_params(knn__n_neighbors=result.best_params_['knn__n_neighbors'],
                       knn__weights=result.best_params_['knn__weights'],
                       knn__metric=result.best_params_['knn__metric'])
pipeline_knn_pca.fit(X_train_undersampled, y_train_undersampled)

#evaluamos en el conjunto de test
pred_test = pipeline_knn_pca.predict(X_test)
f1score = metrics.f1_score(y_test, pred_test)
acc = metrics.accuracy_score(y_test, pred_test)
preci = metrics.precision_score(y_test, pred_test)
recall = metrics.recall_score(y_test, pred_test)

print("Para el modelo optimizado, métricas en el conjunto de test: recall = ", recall, ", F1 = ", f1score, ", acc = ", acc, ", precision = ", preci, )


Fitting 2 folds for each of 600 candidates, totalling 1200 fits
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=uniform;, score=0.696 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=uniform;, score=0.646 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=distance;, score=0.696 total time=   0.0s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, knn__weights=distance;, score=0.646 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=uniform;, score=0.530 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=uniform;, score=0.507 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=distance;, score=0.696 total time=   0.1s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=2, knn__weights=distance;, score=0.646 total time=   0.1s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors

### ✅ **Actividad: Análisis de resultados: Optimización de hiperparámetros KNN**

Ejecute el código de esta notebook y analice los siguientes casos:

1. kNN sin PCA
2. kNN con PCA

Comente sus resultados analizando los valores de recall, f1 score, accuracy y precisión en el conjunto de test.

Cuál es mejor en términos de recall?

Cuál demora menos en entrenar? Vale la pena usar PCA?

Por qué cree que el recall es una métrica más adecuada para este problema? Tip: El recall está relacionado a los falsos negativos. Analice por qué es importante disminuir los falsos negativos en el churn.

### Respuesta: Análisis de resultados

**Métricas en el conjunto de test** (1409 clientes, 374 con churn real):

| Modelo | Recall | F1 | Accuracy | Precision | FN | FP | Mejores hiperparámetros | Tiempo del grid |
|---|---|---|---|---|---|---|---|---|
| KNN base (sin optimizar, k=5) | 0.773 | 0.564 | 0.683 | 0.444 | – | – | default | – |
| KNN optimizado **sin PCA** | **0.840** | 0.572 | 0.666 | 0.434 | **60** | 410 | euclidean, k=100, distance | **≈130 s** |
| KNN optimizado **con PCA** (18 de 40 componentes) | 0.826 | **0.581** | **0.683** | **0.448** | 65 | 381 | manhattan, k=99, uniform | ≈156 s |

*Tiempos medidos para los 1200 fits (600 combinaciones × 2 folds) en una sola máquina; en Colab los valores absolutos cambian, pero la comparación es la misma.*

**¿Cuál es mejor en términos de recall?**
El modelo **sin PCA** (0.840 frente a 0.826). Tiene 60 falsos negativos contra 65: detecta 5 clientes más de los 374 que se van. La diferencia es pequeña. Con solo 2 folds en la CV, la estimación tiene bastante varianza, y eso se nota en que cada modelo eligió hiperparámetros distintos. Los dos modelos optimizados mejoran claramente al KNN base (0.773). La accuracy baja un poco porque la búsqueda optimizó recall y no accuracy.

**¿Cuál demora menos en entrenar?**
En este caso, **sin PCA**. PCA reduce las features de 40 a 18, pero agrega un ajuste de PCA en cada uno de los 1200 fits, y con tan pocos datos (2990 filas balanceadas) ese costo extra pesa más que lo que se ahorra al calcular distancias en menos dimensiones. PCA empezaría a pagar con datasets mucho más grandes o con muchas más dimensiones.

**¿Vale la pena usar PCA?**
**No para este problema.** Da menos recall (la métrica objetivo) y tarda más. Solo gana un poco en precision (+0.014) y F1 (+0.009). Además, muchas de las 40 features son dummies binarias, y la varianza que PCA conserva no es necesariamente la información que separa a los clientes que se van.

**Observación:** en ambos casos el mejor k quedó en el borde del grid (99–100). Eso sugiere que con k > 100 podría mejorar todavía más el recall, y es justo lo que explora la actividad *k vs train recall and val recall* (k hasta 250).

**¿Por qué el recall es la métrica adecuada?**
Recall = TP / (TP + FN), la fracción de los clientes que realmente se van que el modelo logra detectar.
- Un **falso negativo** es un cliente que se va mientras el modelo dice que se queda. No se le ofrece ninguna acción de retención y la empresa pierde todo su ingreso futuro.
- Un **falso positivo** es ofrecerle una promoción a alguien que igual se iba a quedar. El costo es mucho menor (un descuento o una llamada).

Como los costos son asimétricos, conviene minimizar los FN aunque suban los FP. La contraparte es la precision baja (≈0.43): más de la mitad de las alertas son clientes que no se iban a ir. Eso es aceptable siempre que la acción de retención sea barata en comparación con perder al cliente.


# ✅ **Actividad**: Bayesian Hyperparameter Optimization

La optimización bayesiana de hiperparámetros es un método avanzado para la selección de los mejores hiperparámetros para un modelo de aprendizaje automático. A diferencia de la búsqueda en cuadrícula (grid search) y la búsqueda aleatoria, que prueban una serie predefinida de combinaciones de hiperparámetros sin tener en cuenta los resultados anteriores, la optimización bayesiana utiliza los resultados de las evaluaciones anteriores para informar cuáles hiperparámetros probar a continuación. Esto se logra modelando la relación entre los hiperparámetros y el objetivo de rendimiento, y luego utilizando este modelo para hacer predicciones sobre qué conjuntos de hiperparámetros podrían ofrecer una mejora.

Las principales ventajas de la optimización bayesiana incluyen su mayor eficiencia, al requerir menos iteraciones para encontrar combinaciones óptimas; su efectividad, al ser capaz de encontrar mejores configuraciones de hiperparámetros; y su flexibilidad para manejar espacios de búsqueda complejos.

Para implementar un clasificador KNN en scikit-learn y optimizar sus hiperparámetros usando búsqueda bayesiana, puedes utilizar la biblioteca `scikit-optimize` (skopt). Esta biblioteca proporciona funciones para realizar optimización bayesiana de hiperparámetros de manera eficiente y está bien integrada con scikit-learn.



Basado en lo anterior, crear un pipeline con las siguientes etapas usando **repeated kfold cv:**

1. Scaler
2. KNN

Enseguida, pptimizar los param. de KNN usando optimización bayesiana. Finalmente, evaluar en el conjunto de test.

Tip: Ver BayesSearchCV en https://machinelearningmastery.com/scikit-optimize-for-hyperparameter-tuning-in-machine-learning/

In [ ]:
# !pip install scikit-optimize   # descomentar en Colab si skopt no está instalado
import time

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sklearn.metrics as metrics
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer

# repeated k-fold: repite el k-fold con distintas particiones -> estimación del recall con menos varianza
rkfold = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=45)

pipeline_knn_bayes = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# a diferencia de param_grid, aquí se dan RANGOS: el optimizador decide qué puntos probar
search_space = {
    'knn__n_neighbors': Integer(1, 250),  # rango más amplio que el grid (1-100), pues el grid quedó en el borde
    'knn__weights': Categorical(['uniform', 'distance']),
    'knn__metric': Categorical(['euclidean', 'manhattan', 'cosine']),
}

bayes_search = BayesSearchCV(
    pipeline_knn_bayes,
    search_space,
    n_iter=40,          # 40 combinaciones (vs 600 del grid search)
    scoring='recall',
    cv=rkfold,
    random_state=45,
    n_jobs=-1,
)

t0 = time.perf_counter()
bayes_result = bayes_search.fit(X_train_undersampled, y_train_undersampled)
bayes_time = time.perf_counter() - t0

print(f'Tiempo de búsqueda: {bayes_time:.1f} s')
print(f'Mejor recall en validación: {bayes_result.best_score_:.4f} para {dict(bayes_result.best_params_)}')

# BayesSearchCV con refit=True ya reentrena el mejor pipeline con todo el train
best_knn_bayes = bayes_result.best_estimator_
pred_test = best_knn_bayes.predict(X_test)
print("Modelo optimizado con búsqueda bayesiana, métricas en test:",
      f"recall = {metrics.recall_score(y_test, pred_test):.4f},",
      f"F1 = {metrics.f1_score(y_test, pred_test):.4f},",
      f"acc = {metrics.accuracy_score(y_test, pred_test):.4f},",
      f"precision = {metrics.precision_score(y_test, pred_test):.4f}")


### Respuesta: Optimización bayesiana vs Grid Search

| | Grid Search (sin PCA) | Búsqueda bayesiana |
|---|---|---|
| Combinaciones evaluadas | 600 (todas) | **40** |
| Esquema de CV | StratifiedKFold, 2 folds | RepeatedStratifiedKFold, 5 folds × 3 repeticiones |
| Fits totales | 1200 | 600 |
| Rango de k | 1–100 | **1–250** |
| Mejores hiperparámetros | euclidean, k=100, distance | euclidean, **k=238**, distance |
| Recall validación | 0.870 | **0.878** |
| **Recall test** | 0.840 | **0.869** |
| F1 / Acc / Precision test | 0.572 / 0.666 / 0.434 | 0.578 / 0.663 / 0.433 |

**Conclusiones**
- Con **15 veces menos combinaciones**, la búsqueda bayesiana encontró una configuración **mejor en recall de test** (+0.029), y las demás métricas quedaron prácticamente iguales (diferencias ≤ 0.006). Pudo hacerlo porque cada iteración usa los resultados anteriores para decidir dónde probar: concentra las evaluaciones en las zonas prometedoras en vez de recorrerlas todas.
- Al no tener que enumerar todas las combinaciones, se pudo **ampliar el rango de k hasta 250** sin que el costo explote. El grid había quedado en el borde (k=100), y el óptimo resultó estar más arriba.
- El **repeated k-fold** (15 particiones contra 2) da una estimación del recall mucho más estable, así que la elección de hiperparámetros depende menos de cómo cayó una partición en particular.
- Tiempo: ≈25 s con `n_jobs=-1`. No es directamente comparable con el grid, que corrió sin paralelizar, pero hizo la mitad de fits y con una CV más robusta.
- Limitación: la búsqueda bayesiana no garantiza encontrar el óptimo global, y su resultado depende de `n_iter` y de `random_state`.


# ✅ **Actividad: Voronoi 2D y 3D con TSNE (opcional)**

1. **Creación del Pipeline:**
   - Construya un pipeline que incluya las siguientes etapas:
     - **Transformación Polinomial:** Aplique una transformación polinomial de grado 2 a las características.
     - **Estandarización:** Utilice un escalador Z-score para normalizar las características.
     - **Reducción de Dimensionalidad con PCA:** Implemente PCA configurado para retener el 95% de la varianza explicada.
     - **Clasificador:** Utilice el algoritmo K-Nearest Neighbors con los siguientes hiperparámetros 'knn__metric': `'euclidean', 'knn__n_neighbors': 100, 'knn__weights': 'distance'`
2. Genere el Voronoi Tesselation (VT) del clasificador anterior en 2D y 3D. Utilice TSNE como reductor de dimensionalidad para visualizar el VT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy.spatial import Voronoi, cKDTree, voronoi_plot_2d
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import sklearn.metrics as metrics

# ---------- 1. Pipeline ----------
pipeline_vt = Pipeline([
    ('polynomial_features', PolynomialFeatures(degree=2)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('knn', KNeighborsClassifier(metric='euclidean', n_neighbors=100, weights='distance'))
])
pipeline_vt.fit(X_train_undersampled, y_train_undersampled)
print(f"Features: {X_train_undersampled.shape[1]} -> polinomial: "
      f"{pipeline_vt['polynomial_features'].n_output_features_} -> PCA: {pipeline_vt['pca'].n_components_}")

# Se visualiza el conjunto de TEST: con weights='distance', cada punto de train es su propio vecino
# a distancia 0 y el modelo devolvería siempre su etiqueta real (no se vería nada del clasificador).
pred_vt = pipeline_vt.predict(X_test)
print(f"Recall test = {metrics.recall_score(y_test, pred_vt):.4f}")
X_test_pca = pipeline_vt[:-1].transform(X_test)  # todo el pipeline menos KNN: el espacio donde KNN mide distancias
errors = pred_vt != y_test.to_numpy()
CLASS_COLORS = np.array(['#4C72B0', '#DD8452'])  # 0 = se queda, 1 = churn
legend_items = [Line2D([], [], marker='s', ls='', color=CLASS_COLORS[0], label='Predicho: se queda (0)'),
                Line2D([], [], marker='s', ls='', color=CLASS_COLORS[1], label='Predicho: churn (1)'),
                Line2D([], [], marker='x', ls='', color='black', label='Predicción errónea')]

# ---------- 2a. Voronoi 2D ----------
# t-SNE no tiene transformación inversa: no podemos dibujar la frontera "real" de KNN.
# En su lugar, cada muestra de test es una semilla de Voronoi y su celda se pinta con la clase que predijo KNN.
emb_2d = TSNE(n_components=2, random_state=42, init='pca', perplexity=30).fit_transform(X_test_pca)
vor = Voronoi(emb_2d)

fig, ax = plt.subplots(figsize=(10, 8))
for point_idx, region_idx in enumerate(vor.point_region):
    region = vor.regions[region_idx]
    if -1 in region or len(region) == 0:  # celdas abiertas (borde exterior) no se pueden rellenar
        continue
    ax.fill(*vor.vertices[region].T, color=CLASS_COLORS[pred_vt[point_idx]], alpha=0.45)
voronoi_plot_2d(vor, ax=ax, show_points=False, show_vertices=False, line_width=0.3, line_alpha=0.4)
ax.scatter(*emb_2d.T, c=CLASS_COLORS[pred_vt], s=6)
ax.scatter(*emb_2d[errors].T, marker='x', c='black', s=12, linewidths=0.7)
pad = 2
ax.set_xlim(emb_2d[:, 0].min() - pad, emb_2d[:, 0].max() + pad)
ax.set_ylim(emb_2d[:, 1].min() - pad, emb_2d[:, 1].max() + pad)
ax.set_title('Teselación de Voronoi 2D (t-SNE) coloreada por la predicción de KNN (k=100, distance)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(handles=legend_items, loc='upper right')
plt.show()

# ---------- 2b. Voronoi 3D ----------
# Dibujar poliedros de Voronoi en 3D es ilegible. Se aproxima con una malla: cada punto de la malla
# pertenece a la celda de su semilla más cercana (definición de Voronoi) y se dibujan solo los
# puntos de malla donde la clase cambia respecto a un vecino de malla = frontera entre clases.
emb_3d = TSNE(n_components=3, random_state=42, init='pca', perplexity=30).fit_transform(X_test_pca)

GRID_RES = 40
axes_1d = [np.linspace(emb_3d[:, d].min(), emb_3d[:, d].max(), GRID_RES) for d in range(3)]
grid = np.stack(np.meshgrid(*axes_1d, indexing='ij'), axis=-1).reshape(-1, 3)
_, nearest_seed = cKDTree(emb_3d).query(grid)
grid_class = pred_vt[nearest_seed].reshape(GRID_RES, GRID_RES, GRID_RES)

boundary = np.zeros_like(grid_class, dtype=bool)
for axis in range(3):
    change = np.diff(grid_class, axis=axis) != 0
    idx_a = [slice(None)] * 3
    idx_a[axis] = slice(0, -1)
    boundary[tuple(idx_a)] |= change
# solo la frontera cercana a datos reales (la malla también cubre zonas vacías de la caja)
dist_to_seed, _ = cKDTree(emb_3d).query(grid)
near_data = (dist_to_seed < np.percentile(dist_to_seed, 50)).reshape(boundary.shape)
boundary_pts = grid.reshape(GRID_RES, GRID_RES, GRID_RES, 3)[boundary & near_data]

fig = plt.figure(figsize=(11, 9))
ax3 = fig.add_subplot(projection='3d')
ax3.scatter(*boundary_pts.T, c='gray', s=2, alpha=0.15, depthshade=False)
ax3.scatter(*emb_3d.T, c=CLASS_COLORS[pred_vt], s=6, alpha=0.9)
ax3.scatter(*emb_3d[errors].T, marker='x', c='black', s=10, linewidths=0.6)
ax3.set_title('Voronoi 3D (t-SNE): semillas coloreadas por predicción; gris = frontera entre celdas de distinta clase')
ax3.set_xlabel('t-SNE 1')
ax3.set_ylabel('t-SNE 2')
ax3.set_zlabel('t-SNE 3')
ax3.legend(handles=legend_items + [Line2D([], [], marker='o', ls='', color='gray', label='Frontera (malla)')],
           loc='upper left')
plt.show()


### Respuesta: Voronoi 2D y 3D con t-SNE

**Pipeline:** 40 features → `PolynomialFeatures(2)` → 861 → `StandardScaler` → `PCA(0.95)` → **111 componentes** → KNN (euclidean, k=100, distance). **Recall en test = 0.877**, el más alto de todo el notebook.

**Cómo leer los gráficos**
- Cada punto es un cliente de **test** proyectado con t-SNE. Su celda de Voronoi es la región del plano (o del espacio) que está más cerca de él que de cualquier otro punto.
- El color es la clase que **predijo** KNN (azul = se queda, naranja = churn). Las ✕ son errores.
- Se usó test y no train porque, con `weights='distance'`, cada punto de train es su propio vecino a distancia 0, y el modelo simplemente devolvería su etiqueta real.
- En 3D, los poliedros de Voronoi no se pueden leer, así que se aproximaron con una malla 40×40×40. Cada punto de la malla toma la clase de su semilla más cercana, y en gris se muestran solo los puntos donde la clase cambia, que forman la frontera entre regiones.

**Observaciones**
- Hay **regiones grandes y homogéneas de "se queda"** (a la derecha en 2D), compactas y casi sin errores. Son grupos de clientes muy parecidos entre sí y claramente distintos de los que se van, y el modelo los clasifica con confianza.
- La **región de churn** es amplia y está mezclada con celdas azules. Ahí se concentran las ✕. Una ✕ dentro de una celda naranja es un **falso positivo**, y son la mayoría de los errores: es el precio de un recall alto (la precision es baja).
- Las fronteras irregulares y fragmentadas indican que, en la zona central, las dos clases se solapan. Hay clientes que se van y clientes que se quedan con perfiles casi idénticos, y ahí cualquier clasificador basado en vecinos va a cometer errores.

**Advertencia:** t-SNE preserva los **vecindarios locales**, pero no las distancias globales ni la geometría real. La teselación ilustra cómo se agrupan las predicciones; no es la frontera exacta de KNN en las 111 dimensiones. Además, cambiar `perplexity` o `random_state` cambia la forma del gráfico (no el modelo).


# ✅ Actividad: k vs train recall and val recall


1. **Creación del Pipeline:**
   - Construya un pipeline que incluya las siguientes etapas:
     - **Estandarización:** Utilice un escalador Z-score para normalizar las características.
     - **Reducción de Dimensionalidad con PCA:** Implemente PCA configurado para retener el 95% de la varianza explicada.
     - **Clasificador:** Utilice el algoritmo K-Nearest Neighbors.

2. **Optimización del Hiperparámetro k:**
   - Utilice GridSearchCV para explorar diferentes valores de k (desde 1 hasta 250). Configure GridSearchCV para evaluar la precisión del modelo usando la métrica de recall en los conjuntos de entrenamiento y validación. Mantenga fijos todos los demás hiperparámetros.

3. **Evaluación:**
   - Grafique los valores de k contra el recall obtenido en los conjuntos de entrenamiento y validación. Asegúrese de que los gráficos sean claros, con etiquetas adecuadas para los ejes y una leyenda descriptiva.

4. **Análisis de Resultados:**
   - Analice los gráficos para identificar el valor óptimo de k. Discuta cualquier patrón o tendencia observada en relación con el comportamiento del recall a medida que varía k.


In [ ]:
#here your code
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

kfold_k = StratifiedKFold(n_splits=5, shuffle=True, random_state=45)

pipeline_knn_k = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('knn', KNeighborsClassifier())  # weights='uniform', metric='minkowski' (p=2 -> euclidiana) quedan fijos
])

# solo se varía k; el resto de hiperparámetros queda en sus valores por defecto
param_grid_k = {'knn__n_neighbors': np.arange(1, 251, 1)}

search_k = GridSearchCV(pipeline_knn_k, param_grid_k, scoring='recall', cv=kfold_k,
                        return_train_score=True,  # sin esto cv_results_ no guarda el recall de train
                        n_jobs=-1)
search_k.fit(X_train_undersampled, y_train_undersampled)

ks = param_grid_k['knn__n_neighbors']
train_recall = search_k.cv_results_['mean_train_score']
val_recall = search_k.cv_results_['mean_test_score']  # 'test' aquí = fold de validación, no X_test
val_std = search_k.cv_results_['std_test_score']
best_k = search_k.best_params_['knn__n_neighbors']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, train_recall, label='Recall de entrenamiento')
ax.plot(ks, val_recall, label='Recall de validación (5-fold CV)')
ax.fill_between(ks, val_recall - val_std, val_recall + val_std, alpha=0.2, label='Validación ± 1 desv. estándar')
ax.axvline(best_k, color='gray', linestyle='--', label=f'Mejor k = {best_k}')
ax.set_xlabel('k (número de vecinos)')
ax.set_ylabel('Recall')
ax.set_title('KNN (Scaler + PCA 95%): recall de entrenamiento y validación vs k')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f'Mejor k: {best_k} | recall val = {search_k.best_score_:.4f} | recall train = {train_recall[best_k - 1]:.4f}')
for k in (1, 5, 25, 50, 100, 150, 200, 250):
    print(f'k={k:>3}: train = {train_recall[k - 1]:.4f}, val = {val_recall[k - 1]:.4f}, '
          f'brecha = {train_recall[k - 1] - val_recall[k - 1]:.4f}')


### Respuesta: Análisis de k vs recall (Scaler + PCA 95% + KNN, 5-fold CV)

| k | Recall train | Recall val | Brecha (train − val) |
|---|---|---|---|
| 1 | 0.998 | 0.687 | **0.311** |
| 5 | 0.852 | 0.768 | 0.084 |
| 25 | 0.839 | 0.827 | 0.012 |
| 50 | 0.843 | 0.840 | 0.004 |
| 100 | 0.852 | 0.850 | 0.003 |
| 150 | 0.860 | 0.860 | ≈ 0 |
| 250 | 0.866 | 0.865 | 0.001 |

**Mejor k = 243** (recall val = 0.867).

**Patrones observados**
1. **k pequeño → overfitting.** Con k=1 el recall de train es ≈ 1, porque cada punto es su propio vecino más cercano y siempre "acierta". Pero en validación cae a 0.69: el modelo memoriza ruido. La brecha de 0.31 es la firma del overfitting (alta varianza).
2. **Zigzag para k < 30.** Con `weights='uniform'` y k **par** puede haber empate de votos (por ejemplo, 2 contra 2). sklearn desempata eligiendo la clase de menor etiqueta, que es 0 ("se queda"), así que los k pares predicen menos churn y bajan el recall. Con k impar no hay empates en un problema binario.
3. **k ≈ 25–50 → la brecha se cierra.** Train y validación convergen. A partir de ahí el modelo generaliza: lo que mide en train es representativo de lo que hará con datos nuevos.
4. **k > 50 → meseta.** El recall de validación sigue subiendo, pero muy lento (0.840 en k=50 → 0.867 en k=243). La mejora es comparable a la desviación estándar entre folds (banda sombreada, ≈ ±0.02). Dentro de este rango no aparece el underfitting típico (la curva no vuelve a bajar). Un k grande suaviza la frontera de decisión, y en este problema eso favorece al recall. Sería necesario revisar la precision para saber cuánto cuesta esa mejora.

**Valor óptimo de k.** Estrictamente, k=243, pero queda otra vez cerca del borde del rango y su ventaja está dentro del ruido de la CV. Una elección más defendible es **k ≈ 100–150**: tiene prácticamente el mismo recall (0.85–0.86) con una frontera menos suavizada, y no depende de dónde se cortó el rango de búsqueda. Esto coincide con los grid searches anteriores, donde el mejor k también quedó en el límite (99–100), y con la búsqueda bayesiana, que con rango hasta 250 eligió k=238.





# ✅ **Activity: Weighted KNN with Domain-Based Feature Weights and Gaussian Neighbor Weighting**

### 1. **Custom Distance Metric (Domain-Based Feature Weights)**

* Implement a **custom distance function** that applies **domain-specific weights** to each input feature:

  $$
  d(x, x') = \sqrt{\sum_i w_i (x_i - x_i')^2}
  $$
* The function or method must accept:

  * Two input samples $x, x'$
  * An array of feature weights $w = [w_1, w_2, ..., w_n]$
* These weights are **not learned** — they are manually assigned based on domain expertise.


---

### 2. **Gaussian-Based Neighbor Weighting**

* Fix the number of neighbors to **k = 50**.
* Apply a **Gaussian function** to weight each neighbor:

  $$
  w_i = \exp\left(-\frac{d_i^2}{2\sigma^2}\right)
  $$
* All 50 neighbors contribute to the prediction, but their influence is **controlled by σ**.
* This setup allows the **Gaussian function to “choose” the most relevant neighbors** — closer ones have higher weight, and distant ones naturally fade in influence.
* ❗Note that the weights $w_i$ from Gaussian-based neighbor weighting are not the same as those from the domain-specific weights (feature weighting).

---

### 3. **Hyperparameter Optimization**

* Optimize only the **σ (sigma)** parameter.
* Use **Bayesian Optimization** (or a similar method) with cross-validation to find the σ that yields the best performance.

---

### 4. **Evaluation**

* Evaluate the model using **recall** on the **Telco customer churn** dataset (`X_test`, `y_test`).
* Compare your model’s **recall** to a **baseline KNN classifier** using:

  * Euclidean distance
  * Uniform weighting (equal neighbor influence)

---

### 5. **Analysis**

* Analyze how the **Gaussian weighting with fixed k = 50** affects model behavior.
* Reflect on:

  * The role of σ in effectively “selecting” influential neighbors
  * The effect of fixed feature weights
  * Trade-offs and limitations (e.g., sensitivity to σ)
  * Potential extensions, such as adaptive weighting







### 💡 **Hint: Using `BaseEstimator` for Your Custom KNN Classifier**

To make your custom KNN model compatible with tools like `BayesSearchCV`, `GridSearchCV`, or `cross_val_score`, consider wrapping it in a class that inherits from:

* `sklearn.base.BaseEstimator`
* `sklearn.base.ClassifierMixin`

This gives you:

* Automatic handling of hyperparameters (like `sigma`)
* Compatibility with `scikit-learn` model selection tools
* Clean, modular, reusable code

---

### 🧱 Minimal Starting Point

Here’s a simple class to get started:

```python
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np

class GaussianWeightedKNN(BaseEstimator, ClassifierMixin):
    def __init__(self, sigma=1.0, k=50, feature_weights=None):
        self.sigma = sigma
        self.k = k
        self.feature_weights = feature_weights

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        # You will implement the logic for prediction here,
        # using the Gaussian weights on neighbors and weights on features.
        pass
```

Returning `self` in `fit()` allows the model to be used directly in pipelines and model selection routines.

---


### 🧠 Why is the `fit` Method Simple in KNN?

KNN is a **lazy learner**:

* It **does not build an internal model** during training.
* Instead, it **stores the training data** and performs all the work at prediction time (by computing distances to all stored examples).

That’s why the `fit` method just **memorizes** the training data and labels.



In [ ]:
#here your code
# !pip install scikit-optimize   # descomentar en Colab si skopt no está instalado
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted
import sklearn.metrics as metrics
from skopt import BayesSearchCV
from skopt.space import Real


# ---------- 1. Distancia personalizada con pesos por feature ----------
def weighted_euclidean(x: np.ndarray, x_prime: np.ndarray, w: np.ndarray) -> float:
    """Distancia euclidiana ponderada d(x, x') = sqrt(sum_i w_i (x_i - x'_i)^2).

    Args:
        x: Primera muestra, shape (n_features,).
        x_prime: Segunda muestra, shape (n_features,).
        w: Pesos por feature (asignados por conocimiento del dominio), shape (n_features,).

    Returns:
        La distancia ponderada entre x y x'.
    """
    return float(np.sqrt(np.sum(w * (x - x_prime) ** 2)))


def weighted_euclidean_matrix(A: np.ndarray, B: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Versión vectorizada de `weighted_euclidean` para todas las parejas (fila de A, fila de B).

    sum_i w_i (a_i - b_i)^2 == ||sqrt(w)*a - sqrt(w)*b||^2, así que basta con escalar cada feature
    por sqrt(w_i) y usar ||a||^2 + ||b||^2 - 2 a·b. Hacer el doble for con la función escalar
    sería ~4 millones de llamadas por predict.

    Args:
        A: Matriz (n_a, n_features).
        B: Matriz (n_b, n_features).
        w: Pesos por feature, shape (n_features,).

    Returns:
        Matriz de distancias (n_a, n_b).
    """
    sw = np.sqrt(w)
    A_w, B_w = A * sw, B * sw
    sq = (A_w ** 2).sum(axis=1)[:, None] + (B_w ** 2).sum(axis=1)[None, :] - 2 * A_w @ B_w.T
    return np.sqrt(np.maximum(sq, 0.0))  # clip: errores de redondeo pueden dar -1e-15


# sanity check: la versión vectorizada coincide con la definición escalar
_A, _B, _w = np.random.rand(3, 4), np.random.rand(2, 4), np.random.rand(4)
assert np.allclose(weighted_euclidean_matrix(_A, _B, _w)[1, 0], weighted_euclidean(_A[1], _B[0], _w))


# ---------- 2. KNN con pesos gaussianos sobre los vecinos ----------
# ClassifierMixin va a la izquierda de BaseEstimator: es el orden que exige sklearn >= 1.6
# para que los "tags" (es clasificador -> CV estratificada, etc.) se resuelvan bien.
class GaussianWeightedKNN(ClassifierMixin, BaseEstimator):
    """KNN con distancia ponderada por feature y voto ponderado por un kernel gaussiano.

    Cada uno de los k vecinos vota con peso exp(-d^2 / (2 sigma^2)):
    sigma pequeño -> solo cuentan los vecinos muy cercanos (se parece a 1-NN);
    sigma grande -> todos pesan casi igual (se parece a KNN uniforme con k vecinos).

    Args:
        sigma: Ancho del kernel gaussiano.
        k: Número de vecinos considerados (fijo en 50 en esta actividad).
        feature_weights: Pesos de dominio por feature; None = todos 1 (euclidiana normal).
    """

    def __init__(self, sigma: float = 1.0, k: int = 50, feature_weights: np.ndarray | None = None):
        self.sigma = sigma
        self.k = k
        self.feature_weights = feature_weights

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GaussianWeightedKNN":
        # lazy learner: "entrenar" = memorizar los datos
        self.X_train_ = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_, self.y_train_idx_ = np.unique(y, return_inverse=True)
        n_features = self.X_train_.shape[1]
        self.w_ = (np.ones(n_features) if self.feature_weights is None
                   else np.asarray(self.feature_weights, dtype=float))
        if self.w_.shape != (n_features,):
            raise ValueError(f'feature_weights debe tener {n_features} elementos, tiene {self.w_.shape}')
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        check_is_fitted(self, 'X_train_')
        X = np.asarray(X, dtype=float)
        dist = weighted_euclidean_matrix(X, self.X_train_, self.w_)

        # índices de los k vecinos más cercanos de cada muestra (argpartition: O(n), no ordena todo)
        nn_idx = np.argpartition(dist, self.k - 1, axis=1)[:, :self.k]
        nn_dist = np.take_along_axis(dist, nn_idx, axis=1)
        nn_labels = self.y_train_idx_[nn_idx]

        # restar la d^2 mínima de cada fila no cambia qué clase gana (es el mismo factor para
        # todos sus vecinos), pero evita que con sigma chico todos los exp() den 0 -> división 0/0
        d2 = nn_dist ** 2
        gauss_w = np.exp(-(d2 - d2.min(axis=1, keepdims=True)) / (2 * self.sigma ** 2))

        votes = np.zeros((X.shape[0], len(self.classes_)))
        for c in range(len(self.classes_)):
            votes[:, c] = (gauss_w * (nn_labels == c)).sum(axis=1)
        return votes / votes.sum(axis=1, keepdims=True)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ---------- Pesos de dominio (fijos, NO aprendidos) ----------
# Los factores de churn más citados en telecomunicaciones: contrato mes a mes, poca antigüedad,
# cargos altos, fibra óptica, pago con cheque electrónico y no tener servicios de soporte/seguridad.
# Rasgos demográficos como el género no deberían explicar la salida -> peso bajo.
domain_weights = {
    'Contract_Month-to-month': 3.0, 'Contract_Two year': 2.0, 'Contract_One year': 1.5,
    'tenure': 3.0,
    'MonthlyCharges': 2.0, 'TotalCharges': 1.5,
    'InternetService_Fiber optic': 2.0,
    'PaymentMethod_Electronic check': 2.0,
    'OnlineSecurity_No': 1.5, 'TechSupport_No': 1.5,
    'PaperlessBilling': 1.2, 'SeniorCitizen': 1.2,
    'gender': 0.2, 'PhoneService': 0.5,
}
feature_names = X_train_undersampled.columns
feature_weights = np.array([domain_weights.get(col, 1.0) for col in feature_names])
assert set(domain_weights) <= set(feature_names), 'hay un nombre de columna mal escrito en domain_weights'

# ---------- 3. Optimizar solo sigma con búsqueda bayesiana ----------
pipeline_gauss = Pipeline([
    ('scaler', StandardScaler()),  # antes de ponderar: si no, las features en dólares dominan la distancia
    ('gknn', GaussianWeightedKNN(k=50, feature_weights=feature_weights))
])

sigma_search = BayesSearchCV(
    pipeline_gauss,
    {'gknn__sigma': Real(1e-2, 1e2, prior='log-uniform')},  # log: explorar 0.01, 0.1, 1, 10... por igual
    n_iter=30,
    scoring='recall',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=45),
    random_state=45,
    n_jobs=-1,
)
sigma_search.fit(X_train_undersampled, y_train_undersampled)
best_sigma = sigma_search.best_params_['gknn__sigma']
print(f'Mejor sigma: {best_sigma:.4f} | recall val = {sigma_search.best_score_:.4f}')


# ---------- 4. Evaluación en test vs baseline ----------
def test_report(name: str, model: Pipeline) -> dict[str, float]:
    """Calcula las métricas de test de un pipeline ya entrenado y las imprime."""
    pred = model.predict(X_test)
    scores = {'recall': metrics.recall_score(y_test, pred), 'f1': metrics.f1_score(y_test, pred),
              'acc': metrics.accuracy_score(y_test, pred), 'precision': metrics.precision_score(y_test, pred)}
    print(f'{name:<45}', ' '.join(f'{m}={v:.4f}' for m, v in scores.items()))
    return scores


baseline_k50 = Pipeline([('scaler', StandardScaler()),
                         ('knn', KNeighborsClassifier(n_neighbors=50, metric='euclidean', weights='uniform'))])
baseline_k5 = Pipeline([('scaler', StandardScaler()),
                        ('knn', KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='uniform'))])
# ablación: mismo sigma pero sin pesos de dominio, para separar el efecto de cada componente
gauss_no_fw = Pipeline([('scaler', StandardScaler()),
                        ('gknn', GaussianWeightedKNN(sigma=best_sigma, k=50))])

for m in (baseline_k5, baseline_k50, gauss_no_fw):
    m.fit(X_train_undersampled, y_train_undersampled)

test_report('Baseline KNN (euclidean, uniform, k=5)', baseline_k5)
test_report('Baseline KNN (euclidean, uniform, k=50)', baseline_k50)
test_report('Gaussiano k=50, sin pesos de dominio', gauss_no_fw)
test_report('Gaussiano k=50 + pesos de dominio (propuesto)', sigma_search.best_estimator_)

# ---------- 5. Efecto de sigma (para el análisis) ----------
print('\nRecall de validación según sigma (puntos que probó el optimizador):')
res = sigma_search.cv_results_
for s, r in sorted(zip([p['gknn__sigma'] for p in res['params']], res['mean_test_score'])):
    print(f'  sigma = {s:8.4f} -> recall val = {r:.4f}')

# escala de referencia: sigma solo "selecciona" vecinos si es comparable a estas distancias
X_scaled = StandardScaler().fit_transform(X_train_undersampled)
d_train = weighted_euclidean_matrix(X_scaled, X_scaled, feature_weights)
np.fill_diagonal(d_train, np.inf)  # excluir la distancia de cada punto consigo mismo
d_sorted = np.sort(d_train, axis=1)
print(f'\nDistancia mediana al 1er vecino: {np.median(d_sorted[:, 0]):.2f} | '
      f'al vecino 50: {np.median(d_sorted[:, 49]):.2f}')


### Respuesta: KNN con ponderación gaussiana (k = 50) y pesos de dominio por feature

**Resultados en test** (`X_test`, `y_test`):

| Modelo | Recall | F1 | Accuracy | Precision |
|---|---|---|---|---|
| KNN baseline (euclidean, uniform, k=5) | 0.773 | 0.564 | 0.683 | 0.444 |
| KNN baseline (euclidean, uniform, k=50) | 0.818 | 0.589 | 0.697 | 0.460 |
| Gaussiano k=50, **sin** pesos por feature (ablación) | **0.829** | 0.584 | 0.687 | 0.451 |
| **Gaussiano k=50 + pesos de dominio (propuesto)** | 0.816 | **0.596** | **0.707** | **0.470** |

Mejor σ encontrado con optimización bayesiana: **σ ≈ 7.69** (recall de CV = 0.829).

**Rol de σ al "seleccionar" vecinos.** σ solo tiene sentido en relación con las distancias de los datos. Aquí la distancia mediana al 1er vecino es ≈ 2.19 y al 50º es ≈ 4.95. El peso del vecino 50 relativo al 1º es exp(−(4.95² − 2.19²) / 2σ²). Con valores de σ que el optimizador sí probó:

| σ | Peso relativo del vecino 50 | Comportamiento efectivo | Recall CV |
|---|---|---|---|
| 0.84 | ≈ 0.000001 | solo vota el vecino más cercano (≈ 1-NN) | 0.708 |
| 2.06 | ≈ 0.10 | dominan los vecinos cercanos, los lejanos casi no cuentan | 0.814 |
| 7.69 (mejor) | ≈ 0.85 | los 50 votan casi igual (≈ uniforme) | 0.829 |

El recall sube de forma sostenida con σ y luego se aplana para σ ≳ 6 (la curva es *plana* desde ~7 hasta 100). El optimizador nos está diciendo que, **para el recall, conviene que voten los 50 vecinos**. La ponderación gaussiana con σ pequeño se comporta como un k pequeño, y en la actividad *k vs recall* ya vimos que un k pequeño produce overfitting y baja el recall de validación.

**Efecto de los pesos fijos por feature.** Al comparar los dos modelos gaussianos (mismo σ), los pesos de dominio bajaron un poco el recall (0.829 → 0.816), pero mejoraron la precision (0.451 → 0.470), la accuracy (0.687 → 0.707) y el F1 (0.584 → 0.596). Darles más peso a contrato, antigüedad y cargos hace que los vecindarios sean más "puros" respecto a esos factores. Así el modelo da menos falsas alarmas, pero se le escapan algunos clientes que se van y cuyo perfil no sigue esos patrones. El modelo propuesto tiene **el mejor F1, accuracy y precision de todos**, pero no supera en recall al baseline con k=50.

**Trade-offs y limitaciones**
- **σ depende de la escala de las distancias.** Si se reescalan las features o los pesos, el σ óptimo cambia y hay que volver a optimizarlo cada vez que cambia el preprocesamiento.
- **Con k fijo, σ solo puede *reducir* la influencia de los vecinos**, nunca agregar vecinos nuevos. Cuando σ es grande, el modelo colapsa en un KNN uniforme y se vuelve redundante.
- **Los pesos de dominio son una suposición subjetiva.** Nada garantiza que sean óptimos para la métrica objetivo, y aquí costaron recall.
- **Costo computacional:** O(n_test × n_train) distancias por predicción, igual que el KNN estándar.

**Posibles extensiones**
- σ *adaptativo* para cada punto (por ejemplo, σ = distancia al m-ésimo vecino), para que el kernel se ajuste a la densidad local.
- Aprender los pesos por feature (metric learning, como NCA, u optimizarlos junto con σ en la búsqueda bayesiana) en vez de fijarlos a mano.
- Optimizar el umbral de clasificación sobre `predict_proba` para intercambiar precision por recall de forma controlada, ya que el recall es la métrica que importa en churn.
